
# Hedonic regression notebook for bnMAP deals / ПД

Этот ноутбук строит **поэтапную hedonic-модель** для `log(price_sqm)` и специально разделяет блоки:

1. **Рыночная цена**: простой constant-quality proxy через агрегированные средние/медианы по кварталам и субрынкам.  
2. **Geo block**: синтетические гео-признаки в метрах до ключевых объектов + feature engineering / PCA.  
3. **Качество объекта и премия проекта**: извлекаем максимум из bnMAP-признаков и строим shrinkage-оценку project premium.  
4. **Итоговый fit**: финальная regression-модель на `log(price_sqm)`.

Ноутбук старается быть практичным:
- читает **parquet** из `../data/deals_parquet` и `../data/projects_parquet`;
- устойчив к небольшим расхождениям в названиях колонок;
- все синтетические geo-фичи делаются **детерминированно**, чтобы результаты были воспроизводимы;
- модель построена так, чтобы потом ее было легко встроить в Monte Carlo engine как блок `PriceModel`.

## Идея модели

Используем разложение

\[
\log p^{sqm}_{i,r,t}
=
\underbrace{m_{r,t}}_{\text{рыночный уровень}}
+
\underbrace{q_i}_{\text{качество объекта}}
+
\underbrace{g_i}_{\text{geo / accessibility}}
+
\underbrace{\pi_{project(i)}}_{\text{премия проекта}}
+
\varepsilon_i.
\]

Здесь:
- `m_{r,t}` — рыночный блок без макро-факторов;
- `g_i` — synthetic geo block;
- `q_i` — объектное качество по характеристикам лота и корпуса;
- `π_project` — project premium с shrinkage.

## Что важно

- Геоданные пока **синтетические**: они нужны как каркас для будущей замены на реальные расстояния.
- Для Москвы диапазоны расстояний выбраны **правдоподобными**, а генерация зависит от района, класса, масштаба проекта и "урбанистичности".
- Вдохновение для geo feature engineering: использовать accessibility-индексы и затем делать **PCA / orthogonalization**, чтобы убрать лишнюю мультиколлинеарность.


In [ ]:

from __future__ import annotations

import hashlib
import math
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import matplotlib.pyplot as plt

from IPython.display import display
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.linear_model import ElasticNetCV, RidgeCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")
pd.options.display.max_columns = 300
pd.options.display.width = 220
plt.style.use("seaborn-v0_8-whitegrid")
RANDOM_STATE = 42



## 0. Пути к данным

Ниже используется тот же способ загрузки parquet, что и в ноутбуках `sales`:  
поиск файлов в `../data/deals_parquet` и `../data/projects_parquet` с фильтрацией по `REGION_KEY`.


In [ ]:

REGION_KEY = "Москва"

DATA_DIR = Path("..") / "data"
DEALS_DIR = DATA_DIR / "deals_parquet"
PROJECTS_DIR = DATA_DIR / "projects_parquet"


def find_region_parquet_files(folder: Path, region_key: str | None) -> list[Path]:
    files = sorted(folder.glob("*.parquet"))
    if region_key:
        key = region_key.lower()
        files = [p for p in files if key in p.name.lower()]
    return files


DEALS_PARQUET_FILES = find_region_parquet_files(DEALS_DIR, REGION_KEY)
PROJECTS_PARQUET_FILES = find_region_parquet_files(PROJECTS_DIR, REGION_KEY)

if not DEALS_PARQUET_FILES:
    raise FileNotFoundError(f"No deals parquet files for region: {REGION_KEY}")
if not PROJECTS_PARQUET_FILES:
    raise FileNotFoundError(f"No projects parquet files for region: {REGION_KEY}")

print("Region:", REGION_KEY)
print("Deals files:", len(DEALS_PARQUET_FILES))
for p in DEALS_PARQUET_FILES:
    print(" -", p)
print("Projects files:", len(PROJECTS_PARQUET_FILES))
for p in PROJECTS_PARQUET_FILES:
    print(" -", p)



## 1. Общие утилиты чтения и нормализации

Сразу делаем:
- чтение parquet с нормализацией имен колонок;
- мягкое сопоставление названий колонок;
- нормализацию дат, чисел и категорий.


In [ ]:

def normalize_name(x: str) -> str:
    x = str(x).strip().lower()
    replace_map = {
        "ё": "е",
        "\n": " ",
        "\t": " ",
        "  ": " ",
        "«": "",
        "»": "",
        '"': "",
        "'": "",
        "(": "",
        ")": "",
        ",": "",
        ".": "",
        "/": " ",
        "-": " ",
    }
    for a, b in replace_map.items():
        x = x.replace(a, b)
    x = " ".join(x.split())
    return x


def build_norm_map(columns: list[str]) -> dict[str, str]:
    return {normalize_name(c): c for c in columns}


def find_col(columns: list[str], *candidates: str, required: bool = True) -> str | None:
    norm = build_norm_map(columns)
    for cand in candidates:
        key = normalize_name(cand)
        if key in norm:
            return norm[key]
    if required:
        raise KeyError(f"Column not found. Candidates={candidates}")
    return None


def to_datetime_series(series: pd.Series) -> pd.Series:
    x = series.copy()
    x = x.replace({"nan": pd.NA, "None": pd.NA, "": pd.NA})
    parsed = pd.to_datetime(x, errors="coerce", dayfirst=True, utc=True)
    if hasattr(parsed, "dt"):
        parsed = parsed.dt.tz_localize(None)
    num = pd.to_numeric(x, errors="coerce")
    excel_dt = pd.to_datetime(num, unit="D", origin="1899-12-30", errors="coerce")
    return parsed.fillna(excel_dt)


def to_float_series(series: pd.Series) -> pd.Series:
    s = (
        series.astype(str)
        .str.replace("\u00A0", "", regex=False)
        .str.replace(" ", "", regex=False)
        .str.replace(",", ".", regex=False)
        .replace({"nan": np.nan, "None": np.nan, "": np.nan})
    )
    return pd.to_numeric(s, errors="coerce")


def mode_or_nan(x: pd.Series):
    x = x.dropna()
    if x.empty:
        return np.nan
    m = x.mode()
    return m.iloc[0] if not m.empty else x.iloc[0]


def preview_df(df: pd.DataFrame, name: str, n: int = 3):
    print(f"\n{name}: shape={df.shape}")
    display(df.head(n))


DEALS_COLS = [
    "ID проекта",
    "ID корпуса",
    "Проект",
    "Локация",
    "Город",
    "Район",
    "Адрес корпуса",
    "Класс",
    "Девелопер",
    "Застройщик",
    "Конструктив здания",
    "Конструктив",
    "Стадия строительной готовности на дату договора",
    "Старт продаж",
    "Заявленный срок ввода в эксплуатацию",
    "Дата договора",
    "Дата регистрации",
    "ID лота",
    "Тип объекта",
    "Тип объекта недвижимости",
    "Секция",
    "Этаж",
    "Универсальная комнатность",
    "Количество комнат",
    "Площадь согласно ЕГРН",
    "Площадь согласно ПД",
    "Общая проектная площадь с НЛП",
    "Цена за кв. метр",
    "Расчетный бюджет объекта",
    "Отделка по корпусу",
    "Тип ипотеки",
    "Срок в экспозиции до момента сделки, дней",
]

PROJECTS_COLS = [
    "ID проекта",
    "ID корпуса",
    "Проект",
    "Локация",
    "Город",
    "Район",
    "Адрес",
    "Класс",
    "Девелопер",
    "Застройщик",
    "Конструктив",
    "Конструктив здания",
    "Старт продаж",
    "Плановая дата РВЭ",
    "Стадия реализации",
    "Стадия строительства",
    "Секция",
    "Тип договора",
    "Тип сделки",
    "ID лота",
    "Номер объекта в ПД",
    "Тип объекта недвижимости",
    "Тип в декларации",
    "Этаж",
    "Высота потолков мин-макс по корпусу",
    "Кол-во комнат по bnMAP.pro",
    "Кол-во комнат",
    "Общая проектная площадь с НЛП",
    "Участие в сделке",
]


def read_parquet_with_normalized_columns(path: Path, required_cols: list[str]) -> pd.DataFrame:
    schema_cols = pq.ParquetFile(path).schema.names
    by_norm = {str(c).strip(): c for c in schema_cols}
    actual_cols = [by_norm[c] for c in required_cols if c in by_norm]

    part = pd.read_parquet(path, columns=actual_cols if actual_cols else None)
    part = part.rename(columns=lambda c: str(c).strip())

    for c in required_cols:
        if c not in part.columns:
            part[c] = pd.NA
    return part[required_cols]


def read_many_parquet(paths: list[Path], required_cols: list[str], label: str) -> pd.DataFrame:
    frames = []
    for p in paths:
        part = read_parquet_with_normalized_columns(p, required_cols)
        print(f"{label}: {p.name} -> {part.shape}")
        frames.append(part)
    out = pd.concat(frames, axis=0, ignore_index=True)
    print(f"{label} total shape: {out.shape}")
    return out


In [ ]:

deals_raw = read_many_parquet(DEALS_PARQUET_FILES, DEALS_COLS, "deals").copy()
projects_raw = read_many_parquet(PROJECTS_PARQUET_FILES, PROJECTS_COLS, "projects").copy()

projects_raw = projects_raw.drop_duplicates(subset=["ID проекта", "ID лота"]).reset_index(drop=True)

preview_df(deals_raw, "deals_raw")
preview_df(projects_raw, "projects_raw")



## 2. Подготовка сделок и ПД

Здесь мы приводим таблицы к единому каноническому виду, на котором дальше строятся все блоки.


In [ ]:

def prepare_deals(df: pd.DataFrame) -> pd.DataFrame:
    cols = df.columns.tolist()

    out = pd.DataFrame({
        "project_id": df[find_col(cols, "ID проекта")].astype(str),
        "building_id": df[find_col(cols, "ID корпуса", required=False)] if find_col(cols, "ID корпуса", required=False) else pd.Series(index=df.index, dtype="object"),
        "project_name": df[find_col(cols, "Проект")].astype(str),
        "region": df[find_col(cols, "Локация", "Город", required=False)] if find_col(cols, "Локация", "Город", required=False) else pd.Series(index=df.index, dtype="object"),
        "city": df[find_col(cols, "Город", required=False)] if find_col(cols, "Город", required=False) else pd.Series(index=df.index, dtype="object"),
        "district": df[find_col(cols, "Район", required=False)] if find_col(cols, "Район", required=False) else pd.Series(index=df.index, dtype="object"),
        "address": df[find_col(cols, "Адрес корпуса", required=False)] if find_col(cols, "Адрес корпуса", required=False) else pd.Series(index=df.index, dtype="object"),
        "class_raw": df[find_col(cols, "Класс", required=False)] if find_col(cols, "Класс", required=False) else pd.Series(index=df.index, dtype="object"),
        "developer": df[find_col(cols, "Девелопер", required=False)] if find_col(cols, "Девелопер", required=False) else pd.Series(index=df.index, dtype="object"),
        "builder": df[find_col(cols, "Застройщик", required=False)] if find_col(cols, "Застройщик", required=False) else pd.Series(index=df.index, dtype="object"),
        "construction_type": df[find_col(cols, "Конструктив здания", "Конструктив", required=False)] if find_col(cols, "Конструктив здания", "Конструктив", required=False) else pd.Series(index=df.index, dtype="object"),
        "sale_stage": df[find_col(cols, "Стадия строительной готовности на дату договора", required=False)] if find_col(cols, "Стадия строительной готовности на дату договора", required=False) else pd.Series(index=df.index, dtype="object"),
        "sale_start_date": to_datetime_series(df[find_col(cols, "Старт продаж", required=False)]) if find_col(cols, "Старт продаж", required=False) else pd.NaT,
        "rve_plan_date": to_datetime_series(df[find_col(cols, "Заявленный срок ввода в эксплуатацию", required=False)]) if find_col(cols, "Заявленный срок ввода в эксплуатацию", required=False) else pd.NaT,
        "deal_date": to_datetime_series(df[find_col(cols, "Дата договора", required=False)]),
        "registration_date": to_datetime_series(df[find_col(cols, "Дата регистрации", required=False)]) if find_col(cols, "Дата регистрации", required=False) else pd.NaT,
        "lot_id": df[find_col(cols, "ID лота")].astype(str),
        "property_type": df[find_col(cols, "Тип объекта", "Тип объекта недвижимости", required=False)] if find_col(cols, "Тип объекта", "Тип объекта недвижимости", required=False) else pd.Series(index=df.index, dtype="object"),
        "section": df[find_col(cols, "Секция", required=False)] if find_col(cols, "Секция", required=False) else pd.Series(index=df.index, dtype="object"),
        "floor": to_float_series(df[find_col(cols, "Этаж", required=False)]) if find_col(cols, "Этаж", required=False) else pd.Series(np.nan, index=df.index),
        "rooms_raw": df[find_col(cols, "Универсальная комнатность", "Количество комнат", required=False)] if find_col(cols, "Универсальная комнатность", "Количество комнат", required=False) else pd.Series(index=df.index, dtype="object"),
        "area_egrn": to_float_series(df[find_col(cols, "Площадь согласно ЕГРН", required=False)]) if find_col(cols, "Площадь согласно ЕГРН", required=False) else pd.Series(np.nan, index=df.index),
        "area_pd": to_float_series(df[find_col(cols, "Площадь согласно ПД", "Общая проектная площадь с НЛП", required=False)]) if find_col(cols, "Площадь согласно ПД", "Общая проектная площадь с НЛП", required=False) else pd.Series(np.nan, index=df.index),
        "price_sqm": to_float_series(df[find_col(cols, "Цена за кв. метр", required=False)]) if find_col(cols, "Цена за кв. метр", required=False) else pd.Series(np.nan, index=df.index),
        "total_budget": to_float_series(df[find_col(cols, "Расчетный бюджет объекта", required=False)]) if find_col(cols, "Расчетный бюджет объекта", required=False) else pd.Series(np.nan, index=df.index),
        "finishing": df[find_col(cols, "Отделка по корпусу", required=False)] if find_col(cols, "Отделка по корпусу", required=False) else pd.Series(index=df.index, dtype="object"),
        "mortgage_type": df[find_col(cols, "Тип ипотеки", required=False)] if find_col(cols, "Тип ипотеки", required=False) else pd.Series(index=df.index, dtype="object"),
        "exposition_days": to_float_series(df[find_col(cols, "Срок в экспозиции до момента сделки, дней", required=False)]) if find_col(cols, "Срок в экспозиции до момента сделки, дней", required=False) else pd.Series(np.nan, index=df.index),
    })
    return out


def prepare_projects(df: pd.DataFrame) -> pd.DataFrame:
    cols = df.columns.tolist()

    out = pd.DataFrame({
        "project_id": df[find_col(cols, "ID проекта")].astype(str),
        "building_id": df[find_col(cols, "ID корпуса", required=False)] if find_col(cols, "ID корпуса", required=False) else pd.Series(index=df.index, dtype="object"),
        "project_name": df[find_col(cols, "Проект")].astype(str),
        "region": df[find_col(cols, "Локация", "Город", required=False)] if find_col(cols, "Локация", "Город", required=False) else pd.Series(index=df.index, dtype="object"),
        "city": df[find_col(cols, "Город", required=False)] if find_col(cols, "Город", required=False) else pd.Series(index=df.index, dtype="object"),
        "district": df[find_col(cols, "Район", required=False)] if find_col(cols, "Район", required=False) else pd.Series(index=df.index, dtype="object"),
        "address": df[find_col(cols, "Адрес", required=False)] if find_col(cols, "Адрес", required=False) else pd.Series(index=df.index, dtype="object"),
        "class_raw": df[find_col(cols, "Класс", required=False)] if find_col(cols, "Класс", required=False) else pd.Series(index=df.index, dtype="object"),
        "developer": df[find_col(cols, "Девелопер", required=False)] if find_col(cols, "Девелопер", required=False) else pd.Series(index=df.index, dtype="object"),
        "builder": df[find_col(cols, "Застройщик", required=False)] if find_col(cols, "Застройщик", required=False) else pd.Series(index=df.index, dtype="object"),
        "construction_type": df[find_col(cols, "Конструктив", "Конструктив здания", required=False)] if find_col(cols, "Конструктив", "Конструктив здания", required=False) else pd.Series(index=df.index, dtype="object"),
        "sale_start_date": to_datetime_series(df[find_col(cols, "Старт продаж", required=False)]) if find_col(cols, "Старт продаж", required=False) else pd.NaT,
        "rve_plan_date": to_datetime_series(df[find_col(cols, "Плановая дата РВЭ", required=False)]) if find_col(cols, "Плановая дата РВЭ", required=False) else pd.NaT,
        "project_stage": df[find_col(cols, "Стадия реализации", required=False)] if find_col(cols, "Стадия реализации", required=False) else pd.Series(index=df.index, dtype="object"),
        "construction_stage": df[find_col(cols, "Стадия строительства", required=False)] if find_col(cols, "Стадия строительства", required=False) else pd.Series(index=df.index, dtype="object"),
        "section": df[find_col(cols, "Секция", required=False)] if find_col(cols, "Секция", required=False) else pd.Series(index=df.index, dtype="object"),
        "deal_type": df[find_col(cols, "Тип договора", "Тип сделки", required=False)] if find_col(cols, "Тип договора", "Тип сделки", required=False) else pd.Series(index=df.index, dtype="object"),
        "lot_id": df[find_col(cols, "ID лота")].astype(str),
        "lot_num_pd": df[find_col(cols, "Номер объекта в ПД", required=False)] if find_col(cols, "Номер объекта в ПД", required=False) else pd.Series(index=df.index, dtype="object"),
        "property_type": df[find_col(cols, "Тип объекта недвижимости", "Тип в декларации", required=False)] if find_col(cols, "Тип объекта недвижимости", "Тип в декларации", required=False) else pd.Series(index=df.index, dtype="object"),
        "floor": to_float_series(df[find_col(cols, "Этаж", required=False)]) if find_col(cols, "Этаж", required=False) else pd.Series(np.nan, index=df.index),
        "ceiling_raw": df[find_col(cols, "Высота потолков мин-макс по корпусу", required=False)] if find_col(cols, "Высота потолков мин-макс по корпусу", required=False) else pd.Series(index=df.index, dtype="object"),
        "rooms_raw": df[find_col(cols, "Кол-во комнат по bnMAP.pro", "Кол-во комнат", required=False)] if find_col(cols, "Кол-во комнат по bnMAP.pro", "Кол-во комнат", required=False) else pd.Series(index=df.index, dtype="object"),
        "area_pd": to_float_series(df[find_col(cols, "Общая проектная площадь с НЛП", required=False)]) if find_col(cols, "Общая проектная площадь с НЛП", required=False) else pd.Series(np.nan, index=df.index),
        "in_deal": df[find_col(cols, "Участие в сделке", required=False)] if find_col(cols, "Участие в сделке", required=False) else pd.Series(index=df.index, dtype="object"),
    })
    return out


deals = prepare_deals(deals_raw)
projects = prepare_projects(projects_raw)

preview_df(deals, "prepared deals")
preview_df(projects, "prepared projects")


In [ ]:

def normalize_class(x):
    if pd.isna(x):
        return np.nan
    x = normalize_name(str(x))
    if "прем" in x or "элит" in x or "de luxe" in x:
        return "premium"
    if "бизн" in x:
        return "business"
    if "комфорт" in x:
        return "comfort"
    if "эконом" in x or "станд" in x:
        return "economy"
    return x if x else np.nan


def normalize_rooms(x):
    if pd.isna(x):
        return np.nan
    s = normalize_name(str(x))
    if "студ" in s:
        return 0
    if "без типа" in s:
        return np.nan
    digits = "".join(ch for ch in s if ch.isdigit())
    return int(digits) if digits else np.nan


def parse_ceiling_mid(x):
    if pd.isna(x):
        return np.nan
    s = str(x).replace(",", ".")
    parts = s.split("-")
    vals = [pd.to_numeric(p, errors="coerce") for p in parts]
    vals = [v for v in vals if pd.notna(v)]
    if not vals:
        return np.nan
    return float(np.mean(vals))


for df in (deals, projects):
    df["class_group"] = df["class_raw"].map(normalize_class)
    df["rooms"] = df["rooms_raw"].map(normalize_rooms)

projects["ceiling_m"] = projects["ceiling_raw"].map(parse_ceiling_mid)

# Выбираем основную площадь
deals["area_sqm"] = deals["area_egrn"].fillna(deals["area_pd"])
projects["area_sqm"] = projects["area_pd"]

# Фильтрация по региону / типу объекта
if REGION_KEY:
    key = REGION_KEY.lower()
    for df in (deals, projects):
        mask = (
            df["region"].astype(str).str.lower().str.contains(key, na=False)
            | df["city"].astype(str).str.lower().str.contains(key, na=False)
        )
        df.drop(df.index[~mask], inplace=True)

# Только жилые сделки / лоты для hedonic
living_tokens = ["квартира", "апартамент", "жил"]
deals = deals[deals["property_type"].astype(str).str.lower().str.contains("|".join(living_tokens), na=False)].copy()
projects = projects[projects["property_type"].astype(str).str.lower().str.contains("|".join(living_tokens), na=False)].copy()

# Базовые фильтры качества
deals = deals[
    deals["deal_date"].notna()
    & deals["area_sqm"].between(10, 250, inclusive="both")
    & deals["price_sqm"].between(30_000, 3_000_000, inclusive="both")
].copy()

deals["quarter"] = deals["deal_date"].dt.to_period("Q").astype(str)
deals["deal_year"] = deals["deal_date"].dt.year
deals["log_price_sqm"] = np.log(deals["price_sqm"])

projects["sale_start_date"] = to_datetime_series(projects["sale_start_date"])
projects["rve_plan_date"] = to_datetime_series(projects["rve_plan_date"])

print("deals shape after filters:", deals.shape)
print("projects shape after filters:", projects.shape)
display(deals[["project_name", "district", "deal_date", "area_sqm", "price_sqm", "log_price_sqm"]].head())



## 3. Обогащение сделок признаками из ПД

На этом шаге вытаскиваем максимум полезного из `bnMAP`:
- проектный класс;
- конструктив;
- девелопер / застройщик;
- стадия проекта;
- высота потолков;
- дата старта продаж / плановая РВЭ;
- инвентарь по проекту.


In [ ]:

# --- агрегаты по ПД на проектном уровне ---
project_agg = (
    projects.groupby("project_id", as_index=False)
    .agg(
        project_name_pd=("project_name", mode_or_nan),
        district_pd=("district", mode_or_nan),
        class_pd=("class_group", mode_or_nan),
        developer_pd=("developer", mode_or_nan),
        builder_pd=("builder", mode_or_nan),
        construction_type_pd=("construction_type", mode_or_nan),
        project_stage_pd=("project_stage", mode_or_nan),
        construction_stage_pd=("construction_stage", mode_or_nan),
        sale_start_date_pd=("sale_start_date", "min"),
        rve_plan_date_pd=("rve_plan_date", "max"),
        lots_total=("lot_id", "nunique"),
        area_project_total=("area_sqm", "sum"),
        area_project_mean=("area_sqm", "mean"),
        area_project_median=("area_sqm", "median"),
        floor_max_pd=("floor", "max"),
        rooms_mode_pd=("rooms", mode_or_nan),
        ceiling_m_pd=("ceiling_m", "median"),
    )
)

deals = deals.merge(project_agg, on="project_id", how="left")

# --- заполняем пропуски проектными значениями ---
deals["district_final"] = deals["district"].fillna(deals["district_pd"]).fillna("unknown")
deals["class_final"] = deals["class_group"].fillna(deals["class_pd"]).fillna("unknown")
deals["developer_final"] = deals["developer"].fillna(deals["developer_pd"]).fillna("unknown")
deals["builder_final"] = deals["builder"].fillna(deals["builder_pd"]).fillna("unknown")
deals["construction_type_final"] = deals["construction_type"].fillna(deals["construction_type_pd"]).fillna("unknown")

deals["sale_start_date_final"] = deals["sale_start_date"].fillna(deals["sale_start_date_pd"])
deals["rve_plan_date_final"] = deals["rve_plan_date"].fillna(deals["rve_plan_date_pd"])

deals["project_age_months"] = (
    (deals["deal_date"] - deals["sale_start_date_final"]).dt.days / 30.4375
)
deals["months_to_rve"] = (
    (deals["rve_plan_date_final"] - deals["deal_date"]).dt.days / 30.4375
)

deals["project_age_months"] = deals["project_age_months"].clip(-24, 240)
deals["months_to_rve"] = deals["months_to_rve"].clip(-60, 120)

display(
    deals[
        [
            "project_name", "district_final", "class_final", "construction_type_final",
            "lots_total", "area_project_total", "project_age_months", "months_to_rve"
        ]
    ].head()
)



## 4. Блок 1 — рыночная цена

Пока без макро-факторов.  
Берем **простую и устойчивую** оценку рынка через `log(price_sqm)`:

- городской квартальный медианный уровень;
- районный квартальный медианный уровень;
- shrinkage между ними, чтобы маленькие районы/кварталы не шумели.

Итог:
\[
m_{r,t} = w_{r,t}\,\widetilde y_{r,t} + (1-w_{r,t})\,\widetilde y_{city,t},
\qquad
w_{r,t}=\frac{n_{r,t}}{n_{r,t}+k}.
\]


In [ ]:

CITY_SHRINK_K = 25

city_q = (
    deals.groupby("quarter", as_index=False)
    .agg(
        city_log_price_median=("log_price_sqm", "median"),
        city_n=("log_price_sqm", "size"),
    )
)

district_q = (
    deals.groupby(["district_final", "quarter"], as_index=False)
    .agg(
        district_log_price_median=("log_price_sqm", "median"),
        district_n=("log_price_sqm", "size"),
    )
)

deals = deals.merge(city_q, on="quarter", how="left")
deals = deals.merge(district_q, on=["district_final", "quarter"], how="left")

deals["market_weight"] = deals["district_n"] / (deals["district_n"] + CITY_SHRINK_K)
deals["market_log_price"] = (
    deals["market_weight"] * deals["district_log_price_median"]
    + (1.0 - deals["market_weight"]) * deals["city_log_price_median"]
)

deals["market_residual"] = deals["log_price_sqm"] - deals["market_log_price"]

display(
    deals[
        [
            "district_final", "quarter", "log_price_sqm", "city_log_price_median",
            "district_log_price_median", "market_weight", "market_log_price", "market_residual"
        ]
    ].head()
)


In [ ]:

market_diag = (
    deals.groupby("quarter", as_index=False)
    .agg(
        actual=("log_price_sqm", "median"),
        market=("market_log_price", "median"),
        n=("log_price_sqm", "size"),
    )
)
ax = market_diag.plot(x="quarter", y=["actual", "market"], figsize=(12, 4), marker="o", title="Actual vs market block: quarterly median log(price_sqm)")
ax.set_ylabel("log(price_sqm)")
plt.xticks(rotation=45)
plt.show()



## 5. Блок 2 — synthetic geo data in meters

Пока реальных координат нет, поэтому делаем **синтетические расстояния**, но так, чтобы:

1. они были **воспроизводимыми**;
2. они зависели от разумных факторов из bnMAP;
3. диапазоны были **похожи на Москву**.

### Какие расстояния генерируем

- `dist_center_m`
- `dist_metro_m`
- `dist_bus_m`
- `dist_kindergarten_m`
- `dist_school_m`
- `dist_mall_m`
- `dist_park_m`
- `dist_rail_m`
- `dist_hospital_m`

### Принцип

Сначала строим скрытый индекс урбанистичности проекта `urban_score` из:
- районной цены;
- класса;
- масштаба проекта;
- возраста / стадии.

Потом синтетически переводим его в расстояния.  
Чем выше `urban_score`, тем обычно:
- ближе метро / автобус / ритейл;
- ближе центр;
- выше плотность сервисов;
- меньше "автомобильная дистанция".

Это не замена настоящим GIS-данным, а **скелет**, который позже легко заменить.


In [ ]:

def stable_uniform_01(key: str) -> float:
    h = hashlib.sha256(key.encode("utf-8")).hexdigest()
    # 16 hex ~= 64 bits
    x = int(h[:16], 16) / float(16**16 - 1)
    return x


def stable_normal(key: str) -> float:
    # deterministic approx normal via inverse-CDF-free sum of uniforms
    vals = [stable_uniform_01(f"{key}_{i}") for i in range(12)]
    return sum(vals) - 6.0


class_score_map = {"economy": -0.4, "comfort": 0.0, "business": 0.45, "premium": 0.9, "unknown": 0.0}

district_price_level = (
    deals.groupby("district_final")["market_log_price"].median().sort_values()
)
district_price_z = ((district_price_level - district_price_level.mean()) / (district_price_level.std(ddof=0) + 1e-9)).to_dict()

deals["district_price_z"] = deals["district_final"].map(district_price_z).fillna(0.0)
deals["class_score"] = deals["class_final"].map(class_score_map).fillna(0.0)
deals["project_scale_z"] = (
    np.log1p(deals["lots_total"].fillna(deals["lots_total"].median()))
    - np.log1p(deals["lots_total"].fillna(deals["lots_total"].median())).mean()
) / (np.log1p(deals["lots_total"].fillna(deals["lots_total"].median())).std(ddof=0) + 1e-9)

deals["urban_score"] = (
    0.55 * deals["district_price_z"]
    + 0.30 * deals["class_score"]
    + 0.15 * deals["project_scale_z"]
)
deals["urban_score"] = deals["urban_score"].clip(-2.5, 2.5)

def synth_distance(base_min, base_max, urban_score, key, invert=True, extra_shift=0.0):
    # urban_score > 0 => closer if invert=True
    u = stable_uniform_01(key)
    z = stable_normal(key + "_n")
    raw = u ** 1.15
    if invert:
        # higher urban score => smaller quantile
        q = np.clip(raw * np.exp(-0.35 * urban_score + extra_shift), 0.0, 1.0)
    else:
        q = np.clip(raw * np.exp(0.20 * urban_score + extra_shift), 0.0, 1.0)
    val = base_min + q * (base_max - base_min)
    val = val * np.exp(0.08 * z)
    return float(np.clip(val, base_min, base_max))

def make_geo_features(df: pd.DataFrame) -> pd.DataFrame:
    out = pd.DataFrame(index=df.index)
    for idx, row in df.iterrows():
        pid = str(row["project_id"])
        lid = str(row["lot_id"])
        key = f"{pid}|{lid}|{row['district_final']}|{row['class_final']}"
        us = float(row["urban_score"])

        out.loc[idx, "dist_center_m"] = synth_distance(1_000, 35_000, us, key + "|center", invert=True)
        out.loc[idx, "dist_metro_m"] = synth_distance(80, 4_500, us, key + "|metro", invert=True)
        out.loc[idx, "dist_bus_m"] = synth_distance(20, 1_200, us, key + "|bus", invert=True)
        out.loc[idx, "dist_kindergarten_m"] = synth_distance(50, 1_800, us, key + "|kg", invert=True)
        out.loc[idx, "dist_school_m"] = synth_distance(80, 2_500, us, key + "|school", invert=True)
        out.loc[idx, "dist_mall_m"] = synth_distance(150, 8_000, us, key + "|mall", invert=True)
        out.loc[idx, "dist_park_m"] = synth_distance(80, 5_000, us, key + "|park", invert=True)
        out.loc[idx, "dist_rail_m"] = synth_distance(200, 12_000, us, key + "|rail", invert=True)
        out.loc[idx, "dist_hospital_m"] = synth_distance(150, 7_000, us, key + "|hospital", invert=True)
    return out

geo = make_geo_features(deals)
deals = pd.concat([deals, geo], axis=1)

display(deals[[c for c in deals.columns if c.startswith("dist_")]].describe().T)



### 5.1 Feature engineering для geo block

Идея в духе статьи про accessibility:
- расстояния сами по себе использовать можно, но они сильно коррелируют;
- поэтому удобнее построить:
  - лог-расстояния;
  - экспоненциальные accessibility-индексы;
  - PCA-компоненты.

Используем простую гравитационную форму:
\[
A_k(d)=\exp\left(-\frac{d_k}{\lambda_k}\right)
\]
с разумными масштабами \(\lambda_k\) по типу объекта.


In [ ]:

geo_lambdas = {
    "dist_center_m": 12_000,
    "dist_metro_m": 900,
    "dist_bus_m": 350,
    "dist_kindergarten_m": 700,
    "dist_school_m": 900,
    "dist_mall_m": 2_500,
    "dist_park_m": 1_200,
    "dist_rail_m": 2_500,
    "dist_hospital_m": 1_800,
}

geo_dist_cols = list(geo_lambdas.keys())

for col in geo_dist_cols:
    deals[f"log1p_{col}"] = np.log1p(deals[col])
    deals[f"acc_{col.replace('dist_', '').replace('_m', '')}"] = np.exp(-deals[col] / geo_lambdas[col])

# тематические композиты
deals["acc_transit"] = deals[["acc_metro", "acc_bus", "acc_rail"]].mean(axis=1)
deals["acc_family"] = deals[["acc_kindergarten", "acc_school", "acc_hospital"]].mean(axis=1)
deals["acc_retail_green"] = deals[["acc_mall", "acc_park"]].mean(axis=1)
deals["acc_centrality"] = deals["acc_center"]

pca_features = deals[
    [
        "acc_center", "acc_metro", "acc_bus", "acc_kindergarten", "acc_school",
        "acc_mall", "acc_park", "acc_rail", "acc_hospital"
    ]
].copy()

pca_scaler = StandardScaler()
X_geo_scaled = pca_scaler.fit_transform(pca_features)

pca_geo = PCA(n_components=3, random_state=RANDOM_STATE)
geo_pcs = pca_geo.fit_transform(X_geo_scaled)

deals["geo_pc1"] = geo_pcs[:, 0]
deals["geo_pc2"] = geo_pcs[:, 1]
deals["geo_pc3"] = geo_pcs[:, 2]

print("Explained variance ratio:", pca_geo.explained_variance_ratio_)
loadings = pd.DataFrame(
    pca_geo.components_.T,
    index=pca_features.columns,
    columns=["geo_pc1", "geo_pc2", "geo_pc3"],
)
display(loadings.sort_values("geo_pc1", ascending=False))


In [ ]:

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
deals["dist_metro_m"].hist(ax=axes[0, 0], bins=40)
axes[0, 0].set_title("dist_metro_m")
deals["dist_school_m"].hist(ax=axes[0, 1], bins=40)
axes[0, 1].set_title("dist_school_m")
deals["acc_transit"].hist(ax=axes[1, 0], bins=40)
axes[1, 0].set_title("acc_transit")
deals["geo_pc1"].hist(ax=axes[1, 1], bins=40)
axes[1, 1].set_title("geo_pc1")
plt.tight_layout()
plt.show()



## 6. Блок 3 — качество объекта и project premium

Здесь разделяем две вещи:

### 6.1 Качество объекта `q_i`
Это все, что относится к самому лоту / корпусу / проектной оболочке:
- площадь;
- этаж;
- комнатность;
- потолки;
- конструктив;
- класс;
- возраст проекта;
- расстояние до РВЭ;
- тип отделки;
- масштабы проекта.

Идея:
1. убираем из цены рыночный блок `m_{r,t}`;
2. на остатке обучаем регуляризованную модель качества;
3. ее prediction считаем `quality_score`.

### 6.2 Премия проекта `π_project`
После удаления рыночного блока и quality block остаются систематические project-level residuals.
Их усредняем по проекту, но не "в лоб", а с shrinkage:

\[
\pi_j
=
\frac{n_j}{n_j + \tau} \bar r_j.
\]

Это дает устойчивую оценку project premium даже когда у проекта мало сделок.


In [ ]:

# Подготовка базовых numeric / categorical признаков качества
deals["rooms_num"] = deals["rooms"].fillna(deals["rooms"].median())
deals["ceiling_m_final"] = deals["ceiling_m_pd"]
deals["floor_rel"] = deals["floor"] / deals["floor_max_pd"].replace(0, np.nan)
deals["floor_rel"] = deals["floor_rel"].replace([np.inf, -np.inf], np.nan)

quality_num_cols = [
    "area_sqm",
    "floor",
    "floor_rel",
    "rooms_num",
    "ceiling_m_final",
    "project_age_months",
    "months_to_rve",
    "lots_total",
    "area_project_total",
    "area_project_mean",
    "market_log_price",
    "acc_transit",
    "acc_family",
    "acc_retail_green",
    "geo_pc1",
    "geo_pc2",
    "geo_pc3",
]

quality_cat_cols = [
    "class_final",
    "construction_type_final",
    "developer_final",
    "builder_final",
    "district_final",
    "finishing",
]

# Обучаем quality-модель не на полной цене, а на market_residual
y_quality = deals["market_residual"].values

quality_preprocess = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]), quality_num_cols),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("ohe", OneHotEncoder(handle_unknown="ignore", min_frequency=10)),
        ]), quality_cat_cols),
    ]
)

quality_model = Pipeline([
    ("prep", quality_preprocess),
    ("reg", RidgeCV(alphas=np.logspace(-3, 3, 15))),
])

quality_model.fit(deals[quality_num_cols + quality_cat_cols], y_quality)
deals["quality_score"] = quality_model.predict(deals[quality_num_cols + quality_cat_cols])

deals["resid_after_quality"] = deals["market_residual"] - deals["quality_score"]

print("Quality block fitted.")
display(deals[["log_price_sqm", "market_log_price", "market_residual", "quality_score", "resid_after_quality"]].head())


In [ ]:

# Эмпирическая Bayes / shrinkage project premium
PROJECT_PREMIUM_TAU = 20

proj_resid = (
    deals.groupby("project_id", as_index=False)
    .agg(
        project_resid_mean=("resid_after_quality", "mean"),
        project_obs=("resid_after_quality", "size"),
        project_name=("project_name", mode_or_nan),
    )
)
proj_resid["project_premium"] = (
    proj_resid["project_obs"] / (proj_resid["project_obs"] + PROJECT_PREMIUM_TAU)
) * proj_resid["project_resid_mean"]

deals = deals.merge(proj_resid[["project_id", "project_obs", "project_premium"]], on="project_id", how="left")

display(proj_resid.sort_values("project_premium", ascending=False).head(10))
display(proj_resid.sort_values("project_premium", ascending=True).head(10))



## 7. Блок 4 — итоговый fit hedonic regression

Финально фитим модель уже на полной целевой переменной:

\[
\log p^{sqm}_i
=
\beta_0
+ \beta_1 m_{r,t}
+ \beta_2 q_i
+ \beta_3 \pi_{project(i)}
+ \beta^\top z_i
+ \varepsilon_i.
\]

Где `z_i` — дополнительные engineered features.

### Почему так удобно
Такой pipeline хорошо переносится в Monte Carlo:
- рынок можно симулировать отдельно;
- quality score можно считать по характеристикам объекта;
- project premium можно задавать сценарно или фиксировать;
- geo block потом заменяется на реальные GIS-расстояния без переписывания остальной логики.


In [ ]:

final_num_cols = [
    "market_log_price",
    "quality_score",
    "project_premium",
    "area_sqm",
    "floor",
    "floor_rel",
    "rooms_num",
    "ceiling_m_final",
    "project_age_months",
    "months_to_rve",
    "lots_total",
    "area_project_total",
    "area_project_mean",
    "acc_transit",
    "acc_family",
    "acc_retail_green",
    "acc_centrality",
    "geo_pc1",
    "geo_pc2",
    "geo_pc3",
    "dist_center_m",
    "dist_metro_m",
    "dist_park_m",
]

final_cat_cols = [
    "class_final",
    "construction_type_final",
    "developer_final",
    "district_final",
]

feature_cols = final_num_cols + final_cat_cols
target_col = "log_price_sqm"

# Временной split: train на ранних кварталах, test на поздних
quarters_sorted = sorted(deals["quarter"].dropna().unique().tolist())
split_idx = max(1, int(len(quarters_sorted) * 0.7))
train_quarters = set(quarters_sorted[:split_idx])
test_quarters = set(quarters_sorted[split_idx:])

train_mask = deals["quarter"].isin(train_quarters)
test_mask = deals["quarter"].isin(test_quarters)

train_df = deals.loc[train_mask].copy()
test_df = deals.loc[test_mask].copy()

print("train quarters:", sorted(train_quarters))
print("test quarters :", sorted(test_quarters))
print("train shape   :", train_df.shape)
print("test shape    :", test_df.shape)

final_preprocess = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]), final_num_cols),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("ohe", OneHotEncoder(handle_unknown="ignore", min_frequency=10)),
        ]), final_cat_cols),
    ]
)

final_model = Pipeline([
    ("prep", final_preprocess),
    ("reg", ElasticNetCV(
        l1_ratio=[0.05, 0.2, 0.5, 0.8, 0.95],
        alphas=np.logspace(-4, 1, 30),
        cv=5,
        random_state=RANDOM_STATE,
        max_iter=5000,
    )),
])

baseline_pred = np.repeat(train_df[target_col].median(), len(test_df))

final_model.fit(train_df[feature_cols], train_df[target_col])
pred_test = final_model.predict(test_df[feature_cols])

def regression_metrics(y_true, y_pred, name="model"):
    rmse = mean_squared_error(y_true, y_pred) ** 0.5
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    return pd.DataFrame([{
        "model": name,
        "MAE_log": mae,
        "RMSE_log": rmse,
        "R2": r2,
        "MAE_pct_approx": 100 * (np.exp(mae) - 1),
        "RMSE_pct_approx": 100 * (np.exp(rmse) - 1),
    }])

metrics_df = pd.concat([
    regression_metrics(test_df[target_col], baseline_pred, "baseline_median"),
    regression_metrics(test_df[target_col], pred_test, "final_elastic_net"),
], ignore_index=True)

display(metrics_df)


In [ ]:

# Переведем прогноз обратно в цену за кв.м
test_eval = test_df[
    [
        "project_name", "district_final", "quarter", "price_sqm", "market_log_price",
        "quality_score", "project_premium"
    ]
].copy()
test_eval["pred_log_price_sqm"] = pred_test
test_eval["pred_price_sqm"] = np.exp(test_eval["pred_log_price_sqm"])
test_eval["ape"] = (test_eval["pred_price_sqm"] - test_eval["price_sqm"]).abs() / test_eval["price_sqm"]

display(test_eval.head())
print("Median APE:", test_eval["ape"].median())
print("Mean APE  :", test_eval["ape"].mean())


In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(test_df["price_sqm"], test_eval["pred_price_sqm"], alpha=0.4)
mn = min(test_df["price_sqm"].min(), test_eval["pred_price_sqm"].min())
mx = max(test_df["price_sqm"].max(), test_eval["pred_price_sqm"].max())
axes[0].plot([mn, mx], [mn, mx], linestyle="--")
axes[0].set_title("Predicted vs actual price_sqm")
axes[0].set_xlabel("actual")
axes[0].set_ylabel("predicted")

resid = test_df["log_price_sqm"].values - pred_test
axes[1].hist(resid, bins=40)
axes[1].set_title("Test residuals: log scale")
axes[1].set_xlabel("y_true - y_pred")

plt.tight_layout()
plt.show()



## 8. Диагностика блоков по отдельности

Смотрим, сколько вариации объясняет каждый слой:
1. market block;
2. market + quality;
3. market + quality + project premium;
4. final model.


In [ ]:

diag = test_df.copy()

diag["pred_market_only"] = diag["market_log_price"]
diag["pred_market_quality"] = diag["market_log_price"] + diag["quality_score"]
diag["pred_market_quality_project"] = diag["market_log_price"] + diag["quality_score"] + diag["project_premium"]
diag["pred_final"] = pred_test

diag_metrics = pd.concat([
    regression_metrics(diag[target_col], diag["pred_market_only"], "market_only"),
    regression_metrics(diag[target_col], diag["pred_market_quality"], "market+quality"),
    regression_metrics(diag[target_col], diag["pred_market_quality_project"], "market+quality+project"),
    regression_metrics(diag[target_col], diag["pred_final"], "final_model"),
], ignore_index=True)

display(diag_metrics)



## 9. Что потом можно улучшить

### Реальные geo данные
Когда появятся реальные координаты/метро/POI, надо просто заменить блок synthetic geo на:
- GIS distance / travel-time;
- gravity accessibility;
- PCA / factor rotation.

### Рыночный блок
Сейчас он простой: `district-quarter shrinkage median`.
Потом можно сделать:
- hierarchical market index;
- rolling window;
- hedonic time dummy / imputation style constant-quality index.

### Quality block
Сейчас это regularized regression.
Потом можно:
- Mixed Effects / Bayesian partial pooling;
- monotonic constraints;
- explicit interaction terms.

### Project premium
Сейчас empirical Bayes shrinkage.
Потом можно:
- developer random effects;
- building random effects;
- scenario premium for Monte Carlo.

### Для Monte Carlo
На практике из этого ноутбука удобно сохранить:
- fitted `final_model`;
- таблицу `project_premium`;
- таблицу market index по кварталам / субрынкам;
- параметры geo generator или реальные geo-features.


In [ ]:

# Удобный артефакт: словарь готовых таблиц / объектов
artifacts = {
    "market_index_table": deals[
        ["district_final", "quarter", "city_log_price_median", "district_log_price_median", "market_log_price"]
    ].drop_duplicates().reset_index(drop=True),
    "project_premium_table": proj_resid.sort_values("project_obs", ascending=False).reset_index(drop=True),
    "train_metrics": metrics_df,
    "block_diagnostics": diag_metrics,
}

print("Artifacts ready:")
for k, v in artifacts.items():
    if isinstance(v, pd.DataFrame):
        print(f"- {k}: {v.shape}")
    else:
        print(f"- {k}: {type(v)}")
